# Logging and provenance

The library writes to the `sjd` loggers. C++ records use `sjd._core`. Python
records use `sjd.markets.rates`, `sjd.stats`, `sjd.strategy`, and
`sjd.optimise`. `sjd.set_log_level` sets the C++ level and the `sjd` logger
together.

A `Simulation` also keeps the seed and a `metadata` dict, so a number can be
traced back to the build and the inputs that produced it.


In [ ]:
import json
import logging
import sqlite3
import tempfile
from datetime import date
from pathlib import Path

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s")

import numpy as np
import sjd
from sjd import examples, markets, optimise, stats, strategy

sjd.set_log_level("INFO")

market = examples.sample_market(stochastic=True)
environment = market["environment"]
dates = market["simulation_dates"]
anchor = market["anchor"]
portfolio = sjd.Portfolio(examples.sample_contracts(market["forward"], market)["contracts"])
sjd.require_valid(environment)
sjd.require_valid(portfolio, environment)

for name in ("sjd._core", "sjd.markets.rates", "sjd.stats", "sjd.strategy", "sjd.optimise"):
    assert logging.getLogger(name).name == name
print("loggers", 5)


## Levels

`"INFO"` shows the simulation lines. `"WARNING"` keeps warnings and drops
those lines. `"ERROR"` and `"OFF"` drop the warnings as well. The level is
put back to `"INFO"` at the end of the cell.


In [ ]:
class Collect(logging.Handler):
    def __init__(self):
        super().__init__(level=logging.DEBUG)
        self.records = []

    def emit(self, record):
        self.records.append(record)

probe = sjd.VolatilityCurve(
    anchor,
    [date(2026, 12, 16)],
    [examples.FLAT_VOLATILITY],
    allow_extrapolation=True,
)
collected = Collect()
logging.getLogger("sjd").addHandler(collected)

def messages_at(level):
    collected.records.clear()
    sjd.set_log_level(level)
    sjd.simulate(portfolio, environment, dates, paths=2, seed=1, antithetic=False)
    probe.validate()
    return [record.getMessage() for record in collected.records]

try:
    info = messages_at("INFO")
    warning = messages_at("WARNING")
    error = messages_at("ERROR")
    off = messages_at("OFF")
finally:
    sjd.set_log_level("INFO")
    logging.getLogger("sjd").removeHandler(collected)

assert any(message.startswith("simulation start") for message in info)
assert any("Extrapolation past the last pillar is allowed" in message for message in info)
assert not any(message.startswith("simulation start") for message in warning)
assert any("Extrapolation past the last pillar is allowed" in message for message in warning)
assert not any("Extrapolation past the last pillar is allowed" in message for message in error)
assert off == []
print("INFO", len(info))
print("WARNING", len(warning))
print("ERROR", len(error))
print("OFF", len(off))


## Every record

Each line the library writes, triggered once.

| Level | Logger | Record | Trigger |
| --- | --- | --- | --- |
| INFO | `sjd._core` | `simulation start`, `simulation end` | `simulate` |
| INFO | `sjd._core` | `CEM optimisation` | `CurrentExposureMethod(..., optimise=True)` |
| INFO | `sjd.markets.rates` | `loaded market`, including the rates label | `rates_environment` |
| INFO | `sjd.strategy` | `built ... portfolio` | `strategy.portfolio()` |
| INFO | `sjd.stats` | `summary dates=` | `stats.summary` |
| INFO | `sjd.optimise` | `optimiser trial` | `minimise_collar_put_moneyness` |
| INFO | `sjd.optimise` | `optimiser parameters=` | `minimise_strategy` |
| WARNING | `sjd._core` | extrapolation past the last pillar | validating such a curve |
| WARNING | `sjd._core` | flat back-fill to the anchor | validating such a curve |
| WARNING | `sjd._core` | delta and strike conversion | `strike_from_delta` |


In [ ]:
captured = Collect()
logging.getLogger("sjd").addHandler(captured)
try:
    traced = sjd.simulate(portfolio, environment, dates, paths=4, seed=2, antithetic=False)
    sjd.CurrentExposureMethod(
        portfolio,
        sjd.reg_cap_factors("commodity"),
        examples.credit_parameters(),
        True,
    )
    scratch = tempfile.TemporaryDirectory()
    connection = sqlite3.connect(Path(scratch.name) / "pillars.db")
    markets.load_sample(connection)
    markets.rates_environment("XAU", anchor, "USD", connection)
    connection.close()
    strategy.sample_collar(market, market["forward"]).portfolio()
    stats.summary(traced)

    spots = optimise.draw_spot_paths(
        portfolio, environment, dates, paths=4, seed=2, antithetic=False
    )
    discount = [market["discount"].get_discount_factor(day) for day in dates]
    probability = stats.flat_default_probability(anchor, dates, 0.01)
    loss = [0.4] * len(dates)
    optimise.minimise_collar_put_moneyness(
        market,
        market["forward"],
        environment,
        dates,
        spots,
        put_moneyness=[0.96, 0.98],
        call_moneyness=1.03,
        volume=1_000.0,
        discount_factors=discount,
        default_probability=probability,
        loss_given_default=loss,
        objective="cva",
    )
    forward = market["forward"].get_forward(market["collar_settlement"])

    def build(parameters):
        return strategy.CollarStrategy.from_moneyness(
            market["collar_maturity"],
            market["collar_settlement"],
            forward,
            float(parameters[0]),
            1.05,
            1_000.0,
        )

    optimise.minimise_strategy(
        build,
        [0.95],
        environment,
        dates,
        spots,
        discount,
        probability,
        loss,
        bounds=[(0.90, 0.99)],
        objective="total",
        options={"maxiter": 2},
    )
    sjd.VolatilityCurve(
        anchor,
        [date(2026, 12, 16), date(2027, 9, 16)],
        [examples.FLAT_VOLATILITY, examples.FLAT_VOLATILITY],
        allow_extrapolation=True,
    ).validate()
    surface = sjd.VolatilitySurfaceDelta(
        date(2012, 1, 27),
        [
            date(2012, 2, 27),
            date(2012, 3, 27),
            date(2012, 4, 27),
            date(2012, 7, 27),
            date(2013, 1, 27),
            date(2014, 1, 27),
        ],
        [10, 25, 50, 75, 90],
        [[0.18, 0.18, 0.18, 0.20, 0.23, 0.25]] * 5,
        sjd.SurfaceInterpolator.BICUBIC,
    )
    try:
        surface.strike_from_delta(date(2013, 1, 27), -1.0, 100.0)
    except sjd.NumericalError as error:
        assert "delta" in str(error)
        print(type(error).__name__)
    else:
        raise AssertionError("that delta has no strike")
finally:
    logging.getLogger("sjd").removeHandler(captured)

seen = {(record.levelname, record.name, record.getMessage()) for record in captured.records}

def saw(fragment):
    assert any(fragment in message for _, _, message in seen), fragment

for fragment in (
    "simulation start",
    "simulation end",
    "CEM optimisation",
    "loaded market",
    "built collar portfolio",
    "summary dates=",
    "optimiser trial",
    "optimiser parameters=",
    "Extrapolation past the last pillar is allowed",
    "Anchor has no observation",
    "Could not convert a delta to a strike",
):
    saw(fragment)
for level, name, message in sorted(seen):
    print(f"{level:8} {name:22} {message}")


## A file, and a filter

A `logging.FileHandler` on `sjd` receives the same records. Setting
`sjd.optimise` to `ERROR` drops the search lines and leaves `sjd._core` at
`INFO`.


In [ ]:
log_dir = tempfile.TemporaryDirectory()
log_path = Path(log_dir.name) / "sjd.log"
file_handler = logging.FileHandler(log_path, encoding="utf-8")
logging.getLogger("sjd").addHandler(file_handler)
try:
    sjd.simulate(portfolio, environment, dates, paths=2, seed=1, antithetic=False)
finally:
    file_handler.close()
    logging.getLogger("sjd").removeHandler(file_handler)
written = log_path.read_text(encoding="utf-8")
assert "simulation start" in written
assert "simulation end" in written
print(written.strip())

filtered = Collect()
logging.getLogger("sjd").addHandler(filtered)
logging.getLogger("sjd.optimise").setLevel(logging.ERROR)
try:
    sjd.simulate(portfolio, environment, dates, paths=2, seed=1, antithetic=False)
    optimise.minimise_collar_put_moneyness(
        market,
        market["forward"],
        environment,
        dates,
        spots,
        put_moneyness=[0.97],
        call_moneyness=1.03,
        volume=1_000.0,
        discount_factors=discount,
        default_probability=probability,
        loss_given_default=loss,
    )
finally:
    logging.getLogger("sjd.optimise").setLevel(logging.NOTSET)
    logging.getLogger("sjd").removeHandler(filtered)
names = {record.name for record in filtered.records}
assert "sjd._core" in names
assert "sjd.optimise" not in names
print("kept", sorted(names))


## Why the simulation lines arrive together

Path generation releases the GIL. Records written on that side of the call
are queued and flushed when `simulate` returns, so `simulation start` and
`simulation end` appear together rather than one before the paths and one after.


## Debug and trace

`DEBUG` and `TRACE` are compiled out of a Release build, which is what
`pip install` produces. A Debug build can show them. The netting-set decisions
that explain a capital number are `DEBUG`, so a Release run has the
`CEM optimisation` line and not the decisions under it. This notebook prints
the build type and does not invent those lines.


In [ ]:
build_type = sjd.build_info()["build_type"]
print(build_type)
debug_capture = Collect()
logging.getLogger("sjd").addHandler(debug_capture)
try:
    sjd.set_log_level("DEBUG")
    sjd.simulate(portfolio, environment, dates, paths=2, seed=1, antithetic=False)
finally:
    sjd.set_log_level("INFO")
    logging.getLogger("sjd").removeHandler(debug_capture)
if build_type == "Release":
    assert all(record.levelno >= logging.INFO for record in debug_capture.records)


## Where a number came from

`simulation.seed` is the seed that repeats the arrays. `simulation.metadata`
holds the version, the git commit, the build type, the setup and path
timings, the validation warnings, and the inputs. Calling `simulate` again
with `metadata["inputs"]` writes the same arrays.


In [ ]:
metadata = traced.metadata
info = sjd.build_info()
assert metadata["version"] == sjd.__version__
assert metadata["git_commit"] == info["git_commit"]
assert metadata["build_type"] == info["build_type"]
assert metadata["timings"]["setup_ms"] >= 0
assert metadata["timings"]["paths_ms"] >= 0
assert isinstance(metadata["warnings"], list)
assert traced.seed == metadata["inputs"]["seed"]
print(metadata["version"], metadata["build_type"], metadata["git_commit"])
print(metadata["inputs"])

again = sjd.simulate(portfolio, environment, dates, **metadata["inputs"])
assert again.seed == traced.seed
assert np.array_equal(again.spot, traced.spot)
assert np.array_equal(again.mtm, traced.mtm)
print(f"repeated paths {again.paths}")


## Summary and structure costs

`stats.summary` is one row of exposures per simulation date. `metadata=True`
attaches the simulation's provenance. The same flag on `structure_costs`
attaches it to the cost dict. The default results leave it off.


In [ ]:
plain = stats.summary(traced)
assert "metadata" not in plain
with_metadata = stats.summary(traced, metadata=True)
assert with_metadata["metadata"]["inputs"]["seed"] == traced.seed
assert with_metadata["dates"] == dates

exposure_dates = stats.exposure_dates(traced)
costs = stats.structure_costs(
    traced,
    [market["discount"].get_discount_factor(day) for day in exposure_dates],
    stats.flat_default_probability(anchor, exposure_dates, 0.01),
    [0.45] * len(exposure_dates),
)
assert "metadata" not in costs
traced_costs = stats.structure_costs(
    traced,
    [market["discount"].get_discount_factor(day) for day in exposure_dates],
    stats.flat_default_probability(anchor, exposure_dates, 0.01),
    [0.45] * len(exposure_dates),
    metadata=True,
)
assert traced_costs["metadata"]["git_commit"] == info["git_commit"]
assert np.isfinite(traced_costs["total"])
print(f"total {traced_costs['total']:.2f}")


## What can go wrong

A curve that allows extrapolation still runs. The warning is on the curve's
report, in the log, and in `metadata["warnings"]`. The report stays ok.


In [ ]:
forwards, discount_factors, volatilities = examples.fx_curve_values(market["curve_dates"])
wide_forward = sjd.ForwardCurve(
    anchor,
    market["curve_dates"],
    forwards,
    allow_extrapolation=True,
    roll=sjd.ForwardRoll.FORWARD_VOLATILITY,
    volatility=market["volatility"],
)
wide = sjd.RatesEnvironment(anchor)
wide.set_frs(wide_forward)
wide.set_drs(market["discount"])
wide.set_vrs(market["volatility"])
wide.set_fixing_rs(market["fixings"])
curve_report = wide_forward.validate()
assert curve_report.ok
assert any(issue.code == "RATE_SOURCE_EXTRAPOLATION" for issue in curve_report.issues)

wide_book = sjd.Portfolio(examples.sample_contracts(wide_forward, market)["contracts"])
sjd.require_valid(wide_book, wide)
warning_log = Collect()
logging.getLogger("sjd._core").addHandler(warning_log)
try:
    wide_run = sjd.simulate(wide_book, wide, dates, paths=4, seed=1, antithetic=False)
finally:
    logging.getLogger("sjd._core").removeHandler(warning_log)
assert any(
    "Extrapolation past the last pillar is allowed" in record.getMessage()
    for record in warning_log.records
)
assert any("RATE_SOURCE_EXTRAPOLATION" in warning for warning in wide_run.metadata["warnings"])
print(wide_run.metadata["warnings"])
curve_report


## Save and reload

The arrays and the metadata go to a temporary directory. The metadata is
JSON. The arrays are a `.npz` file beside it.


In [ ]:
store = tempfile.TemporaryDirectory()
array_path = Path(store.name) / "paths.npz"
meta_path = Path(store.name) / "metadata.json"
np.savez(array_path, spot=np.asarray(traced.spot), mtm=np.asarray(traced.mtm))
meta_path.write_text(json.dumps(traced.metadata), encoding="utf-8")
restored = json.loads(meta_path.read_text(encoding="utf-8"))
assert restored["inputs"] == traced.metadata["inputs"]
with np.load(array_path) as reloaded:
    assert np.array_equal(reloaded["mtm"], np.asarray(traced.mtm))
print(meta_path.name, array_path.name)
